# 01 · Análisis exploratorio de vuelos

**Objetivo:** entender el dataset de vuelos (15.000 registros) y qué variables se relacionan con la **demora**,
antes de modelar (ver `02_modelado_demoras.ipynb`).

**Datos:** `Vuelos.csv` y `casosNuevos.csv` (ver `data/README.md`). El código de carga, limpieza y gráficos
está en `src/vuelos_ds/`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from vuelos_ds import config, data, eda, plots

plots.set_style()

## 1. Carga y calidad de datos

In [ ]:
df = data.load_vuelos()
casos_nuevos = data.load_casos_nuevos()
print("Vuelos históricos:", df.shape, "| Casos nuevos:", casos_nuevos.shape)
df.info()
nulos = df.isnull().sum()
print("\nNulos:", nulos[nulos > 0].to_dict() if nulos.sum() else "ninguno")

## 2. Análisis univariado

### 2.1 Variables categóricas

In [ ]:
for col in config.VUELOS_CATEGORICAS + config.VUELOS_BOOLEANAS:
    print(f"\n=== {col} ===")
    display(eda.frecuencias(df, col))
    plots.plot_counts(df, col)
    plt.show()

### 2.2 Variables numéricas

In [ ]:
from scipy.stats import skew

for col in config.VUELOS_NUMERICAS + ["puerta_embarque"]:
    print(f"\n=== {col} ===")
    display(df[col].describe().to_frame().T)
    print(f"Asimetría (skewness): {skew(df[col].dropna()):.2f}")
    plots.plot_numeric_overview(df, col)
    plt.show()

### 2.3 Variable objetivo: demora

In [ ]:
plots.plot_target_balance(df)
plt.show()
display(eda.frecuencias(df, config.TARGET))

## 3. Análisis multivariado

### 3.1 Variables numéricas vs. demora

In [ ]:
for col, (tabla, dif) in eda.resumen_por_target(df, config.VUELOS_NUMERICAS).items():
    print(f"\n{col}")
    display(tabla)
    print(f"Diferencia % en la media (demorado vs no demorado): {dif}%")

In [ ]:
for col in config.VUELOS_NUMERICAS:
    plots.plot_numeric_vs_target(df, col)
    plt.show()

### 3.2 Matrices de correlación y covarianza

In [ ]:
cols_matriz = config.VUELOS_NUMERICAS + [config.TARGET]
plots.plot_matrix(df, cols_matriz, "corr")
plt.show()
plots.plot_matrix(df, cols_matriz, "cov")
plt.show()

### 3.3 Relaciones entre variables

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="distancia_vuelo", y="tiempo_estimado_vuelo", hue="demora")
plt.title("Distancia vs tiempo estimado según demora")
plt.show()

In [ ]:
df_rango = eda.agregar_rango_distancia(df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=df, x="condiciones_climaticas", y="visibilidad", ax=ax1)
ax1.set_title("Visibilidad según condiciones climáticas")
sns.boxplot(data=df_rango, x="rango_distancia", y="visibilidad", hue="demora", palette="Set2", ax=ax2)
ax2.set_title("Visibilidad y demora según rango de distancia")
plt.tight_layout()
plt.show()

display(df.groupby("condiciones_climaticas")["visibilidad"].agg(["mean", "std", "median"]).round(2))
display(df_rango.groupby(["rango_distancia", "demora"], observed=True)["visibilidad"].mean().unstack().round(2))

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="congestion_aerea", y="tiempo_estimado_vuelo")
plt.title("Tiempo estimado según congestión aérea")
plt.show()

congestion_por_hora = pd.crosstab(df["hora_salida_programada"], df["congestion_aerea"], normalize="index") * 100
plots.plot_heatmap_table(congestion_por_hora, "Congestión aérea por hora del día (% por hora)",
                         fmt=".0f", cmap="YlOrBr", annot=False)
plt.show()

### 3.4 Probabilidad de demora según cada variable

In [ ]:
for col in ["temporada_alta", "dia_semana", "tipo_avion", "condiciones_climaticas", "congestion_aerea"]:
    print(f"\n=== {col} ===")
    display(eda.tasa_demora_por(df, col).round(2).to_frame("% demora"))
    plots.plot_counts_and_rate(df, col)
    plt.show()

In [ ]:
tabla = df.pivot_table(index="condiciones_climaticas", columns="congestion_aerea",
                       values="demora", aggfunc="mean")
plots.plot_heatmap_table(tabla, "Probabilidad de demora: clima vs congestión aérea")
plt.show()

## 4. Outliers

In [ ]:
vars_outliers = ["distancia_vuelo", "ocupacion_vuelo", "visibilidad", "tiempo_estimado_vuelo"]

fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for ax, col in zip(axes.ravel(), vars_outliers):
    sns.boxplot(x=df[col], color="skyblue", ax=ax)
    ax.set_title(f"Outliers en {col}")
plt.tight_layout()
plt.show()

display(df[vars_outliers].describe())
print("Outliers por método IQR:")
display(eda.outliers_iqr(df, vars_outliers))
print(f"Correlación distancia vs tiempo estimado: {df['distancia_vuelo'].corr(df['tiempo_estimado_vuelo']):.4f}")

In [ ]:
largos = eda.marcar_outliers_distancia(df).rename("es_outlier_distancia")
print("% de demora en vuelos con distancia outlier vs resto:")
display((df.groupby(largos)[config.TARGET].mean() * 100).round(2))

## 5. Conclusiones

*(Valores de la corrida original con los datos reales de la cátedra.)*

- **Desbalance:** 31,85 % de los vuelos se demoran (68,15 % no).
- **Factores con mayor efecto:** el clima (Tormenta 62 %, Niebla 46 %, frente a ~25–27 % con tiempo despejado o
  nublado) y la congestión aérea (Alta 47 %, Media 33 %, Baja 24 %). La temporada alta también pesa (38 % vs 28 %).
- **Visibilidad:** es la única variable numérica que separa los grupos: promedio 13,8 km en demorados vs 16,1 km
  en no demorados (−14,6 %). El resto difiere menos de 2,5 %.
- **Día de la semana y tipo de avión:** efectos chicos (viernes 34 % vs jueves 29 %; Airbus A330 36 % vs A320 30 %).
- **Redundancia:** `distancia_vuelo` y `tiempo_estimado_vuelo` tienen correlación 0,9987; son casi la misma información.
- **Outliers:** 7,3 % de los vuelos tienen distancia atípica (vuelos largos), pero no se demoran más (31,3 % vs 31,9 %).